In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:


from pathlib import Path
OUT = Path("hidden")
OUT.mkdir(parents=True, exist_ok=True)

In [3]:
!pip install -q pyarrow tqdm

import gzip, json, re
from collections import defaultdict
import pandas as pd, pyarrow as pa, pyarrow.parquet as pq
import requests
from tqdm import tqdm

In [4]:
!pip install -q pyarrow tqdm

import gzip, json, re
from collections import defaultdict
import pandas as pd, pyarrow as pa, pyarrow.parquet as pq
import requests
from tqdm import tqdm

In [5]:
# LOCAL, not Colab:
# books[["work_id","book_id_all"]].to_parquet("catalog_ids.parquet", index=False)

ids = pd.read_parquet(OUT / "catalog_ids.parquet")
book_to_work = {str(b): w for w, bs in zip(ids.work_id, ids.book_id_all) for b in bs}
print(f"{len(book_to_work):,} book_ids -> {ids.work_id.nunique():,} works")

1,760,370 book_ids -> 1,002,607 works


In [6]:
URL = "https://mcauleylab.ucsd.edu/public_datasets/gdrive/goodreads/goodreads_reviews_dedup.json.gz"

MAX_PER_WORK = 20
MIN_WORDS, MAX_WORDS = 30, 300
FLUSH_EVERY = 200_000

TAG = re.compile(r"<[^>]+>")
SPOILER = re.compile(r"\*\*\s*spoiler alert\s*\*\*", re.I)

def clean(t):
    t = SPOILER.sub("", TAG.sub(" ", t))
    return re.sub(r"\s+", " ", t).strip()

In [7]:
import pickle

STATE = OUT / "reviews_state.pkl"
SHARDS = OUT / "review_shards"
SHARDS.mkdir(exist_ok=True)

if STATE.exists():
    s = pickle.loads(STATE.read_bytes())
    kept = defaultdict(int, s["kept"])
    n_seen, n_kept, shard = s["n_seen"], s["n_kept"], s["shard"]
    print(f"resuming: skipping {n_seen:,} lines, {n_kept:,} already kept")
else:
    kept, n_seen, n_kept, shard = defaultdict(int), 0, 0, 0

resume_at = n_seen
buf = []

def save():
    global buf, shard
    if buf:
        pd.DataFrame(buf).to_parquet(SHARDS / f"{shard:04d}.parquet",
                                     index=False, compression="zstd")
        shard += 1
        buf = []
    STATE.write_bytes(pickle.dumps(
        {"kept": dict(kept), "n_seen": n_seen, "n_kept": n_kept, "shard": shard}))

In [8]:
with requests.get(URL, stream=True, timeout=120) as r:
    r.raise_for_status()
    total = int(r.headers.get("Content-Length", 0))
    bar = tqdm(total=total or None, unit="B", unit_scale=True, unit_divisor=1024)

    raw, _read = r.raw, r.raw.read
    def counting_read(n):
        chunk = _read(n)
        bar.update(len(chunk))
        return chunk
    raw.read = counting_read

    for i, line in enumerate(gzip.GzipFile(fileobj=raw)):
        if i < resume_at:
            continue
        n_seen += 1
        rec = json.loads(line)

        w = book_to_work.get(str(rec.get("book_id")))
        if w is None or kept[w] >= MAX_PER_WORK:
            continue

        txt = clean(rec.get("review_text") or "")
        if not MIN_WORDS <= len(txt.split()) <= MAX_WORDS:
            continue

        buf.append({"work_id": w, "rating": rec.get("rating", 0), "text": txt})
        kept[w] += 1
        n_kept += 1

        if len(buf) >= FLUSH_EVERY:
            save()

    bar.close()

save()
print(f"done — seen {n_seen:,}  kept {n_kept:,}  works {len(kept):,}")

100%|██████████| 4.98G/4.98G [16:18<00:00, 5.46MB/s]


done — seen 15,739,967  kept 3,755,257  works 896,632


In [9]:
shards = sorted(SHARDS.glob("*.parquet"))
rev = pd.concat([pd.read_parquet(p) for p in shards], ignore_index=True)
rev.to_parquet(OUT / "reviews_sample.parquet", index=False, compression="zstd")
print(rev.shape, f"{(OUT/'reviews_sample.parquet').stat().st_size/1e6:.0f} MB")

(3755257, 3) 995 MB


In [10]:
n = rev.groupby("work_id").size()
print(n.describe())
for k in [1, 5, 10, 20]:
    print(f">= {k:>2}: {n.ge(k).sum():,} works")

count    739828.000000
mean          5.075851
std           5.960757
min           1.000000
25%           1.000000
50%           2.000000
75%           6.000000
max          20.000000
dtype: float64
>=  1: 739,828 works
>=  5: 234,426 works
>= 10: 129,991 works
>= 20: 66,405 works


In [12]:
sample = rev[rev.work_id.isin(n[n >= 5].index)].sample(5, random_state=0)

for _, r in sample.iterrows():
    print(f"--- work_id {r.work_id}  [{r.rating}]")
    print(r.text[:500])
    print()

--- work_id 41056023  [5]
We are proud to announce that THE SECRET WORLD OF CHRISTOVAL ALVAREZ by Ann Swinfen is a B.R.A.G.Medallion Honoree. This tells a reader that this book is well worth their time and money!

--- work_id 44817914  [5]
*A Copy of this Book Was Given To Me By The Author In Exchange For An Honest Review* THE NINES is an ongoing romantic suspense series. Each book can be read as a STAND ALONE NOVEL or as PART OF THE SERIES. Anyone like me will quickly identify with the main characters in The Nines and wish them every success in their task to secure justice for the literally hundreds of rape victims. You will realize that going to the police is not an option as one perpetrator is the son of the police chief and an

--- work_id 6767866  [5]
A beautiful romance that will take your breath away. Elizabeth Lowell has a way with words that makes you feel every emotion. Only Mine is a story that stayed with me long after I read the last page. A one click buy. If you love hist

In [13]:
w = n[n >= 10].index[0]
print(f"work_id {w}")
for _, r in rev[rev.work_id == w].iterrows():
    print(f"\n[{r.rating}] {r.text[:400]}")

work_id 1000099

[2] Sekavahko naytelma, josta ie kunnolla ota selvaa mita tapahtuu ja missa. hahmojen suhteet ovat sekavat ja vahan valia pitaa katsoa henkiloluettelosta etta kuka oli kukakin. En pitanyt, ainoa kiinnostava henkilo oli Masa.

[3] Once again I was surprised to read a book by a Russian author with very realistic characters. People are just the same even in other cultures and time periods. There is satirical humor and there is disappointment. I've found that the Russian authors I have read are able to portray humanity in such a real and universal way.

[3] "Tenemos la obligacion de vivir" La forma en la que se retrata la vida de los personajes deja en el aire varias preguntas sobre la existencia de la felicidad y el sentido de la vida que quedan en la garganta del lector atragantadas de forma terrible e ineludible. Una excelente obra de arte con la que hubiera conectado mejor en un estado melancolico de la vida.

[3] 2 1/2 stars, maybe? I only read it. I haven't seen/hear

In [18]:
import re
from tqdm.auto import tqdm
tqdm.pandas()

COMMON = set("the a an and or but is was were to of in it that this for with as on".split())

def en_frac(t):
    w = re.findall(r"[a-z']+", t.lower())
    return sum(x in COMMON for x in w) / max(len(w), 1)

before = len(rev)
rev["en_frac"] = rev.text.progress_map(en_frac)

print(rev.en_frac.describe())
print(f"would drop {(rev.en_frac <= 0.08).sum():,} of {before:,} "
      f"({(rev.en_frac <= 0.08).mean():.1%})")

  0%|          | 0/3755257 [00:00<?, ?it/s]

count    3.755257e+06
mean     2.816509e-01
std      6.853320e-02
min      0.000000e+00
25%      2.553191e-01
50%      2.884615e-01
75%      3.207547e-01
max      6.627907e-01
Name: en_frac, dtype: float64
would drop 117,364 of 3,755,257 (3.1%)


In [19]:
rev = rev[rev.en_frac > 0.08].drop(columns="en_frac")
print(f"{len(rev):,} reviews, {rev.work_id.nunique():,} works")

3,637,893 reviews, 734,047 works


In [20]:
# ok but does reviews also actually carry the signal we need?
MOOD = ["cozy", "heartwarming", "tearjerker", "cried", "page-turner",
        "couldn't put it down", "slow burn", "dragged", "one sitting",
        "comfort read", "dark", "bleak", "funny", "wholesome"]

for m in MOOD:
    hits = rev.text.str.contains(m, case=False, regex=False)
    print(f"{m:<22} {hits.sum():>8,}  ({hits.mean():.2%})")

cozy                     18,207  (0.50%)
heartwarming             13,298  (0.37%)
tearjerker                  546  (0.02%)
cried                    18,724  (0.51%)
page-turner               6,842  (0.19%)
couldn't put it down     24,718  (0.68%)
slow burn                 3,741  (0.10%)
dragged                  15,716  (0.43%)
one sitting              18,920  (0.52%)
comfort read                900  (0.02%)
dark                    136,633  (3.76%)
bleak                     6,464  (0.18%)
funny                   130,726  (3.59%)
wholesome                 1,850  (0.05%)


In [21]:
for m in ["cozy", "cried", "one sitting", "couldn't put it down", "funny"]:
    w = rev.loc[rev.text.str.contains(m, case=False, regex=False), "work_id"].nunique()
    print(f"{m:<22} {w:>7,} works")

cozy                    10,007 works
cried                   15,526 works
one sitting             17,660 works
couldn't put it down    22,490 works
funny                   80,863 works


In [24]:
rev.to_parquet(OUT / "reviews_en.parquet", index=False, compression="zstd")
print(len(rev), rev.work_id.nunique())

3637893 734047


## note

this is where i switch to gpu

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import numpy as np, pandas as pd
OUT = Path("hidden")

Mounted at /content/drive


In [2]:
!pip install -q sentence-transformers
import torch
from sentence_transformers import SentenceTransformer
print(torch.cuda.get_device_name(0))

Tesla T4


In [3]:
rev = pd.read_parquet(OUT / "reviews_en.parquet")
print(f"{len(rev):,} reviews, {rev.work_id.nunique():,} works")

3,637,893 reviews, 734,047 works


In [5]:
from tqdm.auto import tqdm
tqdm.pandas()

docs = (rev.sort_values(["work_id", "rating"], ascending=[True, False])
        .groupby("work_id").text
        .progress_apply(lambda s: " ".join(s.head(20)))
        .rename("text").reset_index())
print(f"{len(docs):,} works")

  0%|          | 0/734047 [00:00<?, ?it/s]

734,047 works


In [7]:
from pathlib import Path
CHUNKS = OUT / "emb_chunks"
CHUNKS.mkdir(exist_ok=True)

model = SentenceTransformer("all-MiniLM-L6-v2", device="cuda")

SIZE = 20_000
texts = docs.text.tolist()

for start in range(0, len(texts), SIZE):
    p = CHUNKS / f"{start:07d}.npy"
    if p.exists():
        continue
    e = model.encode(texts[start:start + SIZE], batch_size=256,
                     show_progress_bar=True, normalize_embeddings=True,
                     convert_to_numpy=True).astype("float32")
    np.save(p, e)
    print(f"saved {p.name}  {start + len(e):,}/{len(texts):,}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/79 [00:00<?, ?it/s]

saved 0000000.npy  20,000/734,047


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

saved 0020000.npy  40,000/734,047


...

saved 0680000.npy  700,000/734,047


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

saved 0700000.npy  720,000/734,047


Batches:   0%|          | 0/55 [00:00<?, ?it/s]

saved 0720000.npy  734,047/734,047


In [8]:
parts = sorted(CHUNKS.glob("*.npy"))
emb = np.concatenate([np.load(p) for p in parts])
assert len(emb) == len(docs)

np.save(OUT / "review_emb.npy", emb)
docs[["work_id"]].to_parquet(OUT / "review_emb_ids.parquet", index=False)
print(emb.shape, f"{emb.nbytes/1e6:.0f} MB")

(734047, 384) 1127 MB


In [9]:
!pip install -q faiss-cpu
import faiss

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
faiss.write_index(index, str(OUT / "reviews.faiss"))
print(index.ntotal)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 40.1 MB/s eta 0:00:00
734047


In [10]:
q = model.encode(["feel-good and short"], normalize_embeddings=True)
D, I = index.search(q.astype("float32"), 10)
docs.iloc[I[0]].work_id.tolist()

['14415988',
 '18934079',
 '25518586',
 '24757153',
 '18104716',
 '26960173',
 '40232092',
 '41935447',
 '39852282',
 '56214358']

In [12]:
q = model.encode(["heartwarming and uplifting, made me smile"],
                 normalize_embeddings=True)
D, I = index.search(q.astype("float32"), 50)

res = docs.iloc[I[0]][["work_id"]].copy()
res["score"] = D[0]
res.to_csv(OUT / "probe.csv", index=False)

In [13]:
q = model.encode(["heartwarming and uplifting, made me smile"],
                 normalize_embeddings=True)
D, I = index.search(q.astype("float32"), 50)
print(docs.iloc[I[0]].work_id.tolist())

['41123963', '15715784', '23680005', '25047261', '2491159', '47994361', '50768109', '2345432', '51772780', '20202133', '23667343', '47546928', '19989227', '26532158', '243189', '44460068', '56222685', '54798356', '51437032', '43218612', '58011634', '20388785', '45739878', '55646459', '26616296', '283173', '21361766', '9641604', '749945', '1131880', '43129327', '1465147', '795040', '44406252', '56217230', '42678094', '9497207', '6667329', '70607', '52948062', '48646097', '23682256', '10545803', '45205131', '21938169', '44318468', '41353386', '49192153', '691838', '18584545']


In [14]:
for qtext in ["I finished this with a huge smile on my face",
              "this book made me cry",
              "I could not put it down, read it in one sitting"]:
    q = model.encode([qtext], normalize_embeddings=True)
    D, I = index.search(q.astype("float32"), 20)
    print(f"\n=== {qtext}")
    print(docs.iloc[I[0]].work_id.tolist())


=== I finished this with a huge smile on my face
['44049882', '53381651', '57839505', '40932347', '51821659', '1356883', '24065803', '681661', '5178162', '243189', '22001134', '39905370', '44845172', '26670001', '42850631', '5268320', '43116554', '45382527', '39649964', '25696366']

=== this book made me cry
['3056025', '13840585', '41054167', '2668555', '41358718', '14320962', '44863150', '52690063', '51774491', '52427707', '6979575', '42244082', '57294744', '48165', '13582894', '22429918', '5764282', '1407081', '14550465', '6365653']

=== I could not put it down, read it in one sitting
['42276036', '425501', '49944026', '4055610', '10454913', '48778176', '14614719', '21358640', '43125726', '684187', '22296522', '48161814', '26427973', '27735736', '964360', '802709', '46608499', '42075943', '17754385', '1199368']


In [15]:
w = docs.work_id.iloc[0]
d = docs[docs.work_id == w].text.iloc[0]
print(len(d.split()), "words")

223 words


In [16]:
n = rev.groupby("work_id").size()
good = set(n[n >= 5].index)
mask = docs.work_id.isin(good).to_numpy()

idx2 = faiss.IndexFlatIP(emb.shape[1])
idx2.add(emb[mask])
sub = docs[mask].reset_index(drop=True)

q = model.encode(["I finished this with a big smile on my face"],
                 normalize_embeddings=True)
D, I = idx2.search(q.astype("float32"), 20)
print(sub.iloc[I[0]].work_id.tolist())

['16957329', '18967965', '577668', '4208329', '17316785', '25475795', '16943257', '28096028', '25784612', '25487896', '49322086', '23645510', '26021888', '42470883', '13483419', '42916768', '18217235', '18265117', '47207025', '26917226']


In [17]:
D, I = idx2.search(q.astype("float32"), 500)
print(sub.iloc[I[0]].work_id.tolist()[:500])

['16957329', '18967965', '577668', '4208329', '17316785', '25475795', '16943257', '28096028', '25784612', '25487896', '49322086', '23645510', '26021888', '42470883', '13483419', '42916768', '18217235', '18265117', '47207025', '26917226', '25374301', '17438159', '19140994', '25698990', '21557504', '45585402', '42241284', '47202366', '23684134', '3473940', '23912367', '39834830', '21982945', '26655068', '15970676', '21989535', '25490321', '21553745', '48485', '25315302', '11247213', '20734075', '4493959', '39901452', '17847847', '52113307', '2884461', '2018017', '51817045', '22592363', '48548608', '25441722', '86125', '15223263', '42032954', '13009809', '26637637', '45380778', '20654276', '52297984', '40429780', '43193195', '25747377', '49616101', '25432110', '44235417', '43072768', '675039', '18537958', '45125374', '54810218', '50687269', '3147419', '1420167', '44075664', '18231136', '25875621', '44611111', '45432672', '41787820', '16943099', '674566', '26414536', '44689845', '53090016'